In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    average_precision_score, classification_report, confusion_matrix, f1_score
)

RANDOM_STATE = 2137

In [2]:
df = pd.read_csv("data_for_classifier.csv")
df = df.drop(columns = ["datetime", "ip_subnet", "postcode"])
df = df.loc[:, ~df.columns.str.startswith("merchant_info.")]
categorical_cols = ["type", "category", "payment_method", "device"]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

In [3]:
split_idx = int(len(df) * 0.9)
train_full = df.iloc[:split_idx]
test = df.iloc[split_idx:]

val_split_idx = int(len(train_full) * (8/9))
train = train_full.iloc[:val_split_idx]
val = train_full.iloc[val_split_idx:]

In [4]:
features = [col for col in df.columns if col not in ["is_fraud", "orig", "dest", "city_state"]]
binary_features = [col for col in features if df[col].isin([0, 1]).all()]
continuous_features = [col for col in features if col not in binary_features]
preprocessor = ColumnTransformer(
    transformers=[("num", StandardScaler(), continuous_features)], 
    remainder="passthrough" 
)

In [5]:
counts = train.groupby(["city_state", "is_fraud"]).size().unstack(fill_value=0)
valid_silos = counts[(counts[0] >= 5) & (counts[1] >= 5)].index
print(f"Found {len(valid_silos)} valid city_state silos.")

Found 6 valid city_state silos.


In [6]:
global_coef = np.zeros((1, len(features)))
global_intercept = np.zeros(1)
total_samples = 0

for silo in valid_silos:
    silo_mask = train["city_state"] == silo
    X_local = train.loc[silo_mask, features]
    y_local = train.loc[silo_mask, "is_fraud"]
    
    local_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("logistic_regression", LogisticRegression(
            class_weight="balanced", 
            max_iter=1000, 
            random_state=RANDOM_STATE
        ))
    ])
    
    local_pipeline.fit(X_local, y_local)
    local_model = local_pipeline.named_steps["logistic_regression"]
    
    n = len(X_local)
    global_coef += local_model.coef_ * n
    global_intercept += local_model.intercept_ * n
    total_samples += n

global_coef /= total_samples
global_intercept /= total_samples

In [7]:
X_train_full = train_full[features]
preprocessor.fit(X_train_full)

X_val = preprocessor.transform(val[features])
y_val = val["is_fraud"]

X_test = preprocessor.transform(test[features])
y_test = test["is_fraud"]

In [8]:
global_model = LogisticRegression()
global_model.classes_ = np.array([0, 1])
global_model.coef_ = global_coef
global_model.intercept_ = global_intercept

In [9]:
val_scores = global_model.predict_proba(X_val)[:, 1]
thresholds = np.unique(np.quantile(val_scores, np.linspace(0, 1, 501)))
val_f1 = [f1_score(y_val, val_scores >= threshold) for threshold in thresholds]
best_threshold = float(thresholds[int(np.argmax(val_f1))])

print(f"Global Validation-selected threshold: {best_threshold:.6f}")
print(f"Global Validation F1: {max(val_f1):.4f}")

Global Validation-selected threshold: 0.588312
Global Validation F1: 0.6786


In [10]:
test_scores = global_model.predict_proba(X_test)[:, 1]
y_pred = (test_scores >= best_threshold).astype(int)

print("Federated Test classification report:")
print(classification_report(y_test, y_pred))

Federated Test classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    108833
           1       0.61      0.99      0.75       185

    accuracy                           1.00    109018
   macro avg       0.80      0.99      0.88    109018
weighted avg       1.00      1.00      1.00    109018

